# Group 2: feature processing notebook

Этот ноутбук предназначен для исследовательской обработки признаков группы `2`.

Логика ячеек специально повторяет структуру `src/features/group_2/feature_processor.py`,
чтобы позже перенос был почти механическим.

In [1]:
from pathlib import Path
import pandas as pd
import sys

# путь до корня проекта
PROJECT_ROOT = Path("../../").resolve()

# добавляем в PYTHONPATH
sys.path.append(str(PROJECT_ROOT))

# теперь обычный импорт
from src.utils.spec_converter import create_feature_spec_template
from src.utils.io import load_feature_names_from_txt

In [2]:
# Пути относительно папки notebooks/group_2/
DATA_PATH = Path("../../data/raw/MIPT_hackathon_dataset.csv")
FEATURES_PATH = Path("../../data/feature_groups/features_group_2.txt")
OUTPUT_DIR = Path("../../notebook_outputs/group_2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_csv(DATA_PATH)
feature_names = load_feature_names_from_txt(FEATURES_PATH)
block_df = df[feature_names].copy()

print("Block shape:", block_df.shape)
display(block_df.head())

Block shape: (5383, 19)


,lead_LEADQUALIFYCATION,contact_Код ПВЗ,lead_Вес (грамм)*,lead_Объявленная ценность (руб),lead_будущие покупки,lead_Сумма наложенного платежа (руб),lead_ACTUAL-FORMAT,lead_responsible_user_id,lead_Тариф Доставки,lead_Линейная длина (см),lead_utm_content,lead_Дата создания сделки,contact_LTV,sale_date,lead_LTV,lead_FORMNAME,outcome_unknown,lead_loss_reason_id,lead_Дата получения денег на Р/С
0,NaN,SPB140,352.0,NaN,не известно,NaN,NaN,MGR_0007,Посылка склад-склад,NaN,NaN,1.761858e+09,10077.0,2025-11-01,NaN,NaN,False,NaN,1.763327e+09
1,a,YEVP3,1386.0,NaN,не известно,NaN,NaN,MGR_0023,Посылка склад-склад,NaN,704258743_5668147522_17363919476_type1_mobile_...,1.761944e+09,15780.0,2025-11-01,NaN,consult,False,NaN,1.763413e+09
2,NaN,BRN10,1368.0,NaN,не известно,NaN,NaN,MGR_0023,Посылка склад-склад,NaN,NaN,NaN,97999.0,2025-11-01,79498.0,NaN,False,NaN,1.763327e+09
3,NaN,DON2,688.0,NaN,не известно,NaN,NaN,MGR_0023,Посылка склад-склад,NaN,NaN,1.761944e+09,NaN,2025-11-01,NaN,NaN,False,NaN,NaN
4,NaN,NaN,672.0,NaN,матрас,NaN,NaN,MGR_0002,Посылка склад-дверь,NaN,703798483_5657074625_17331454991_type1_mobile_...,1.761944e+09,8495.0,2025-11-01,NaN,artraid-dem,False,NaN,1.763068e+09


## Функции обработки признаков

Названия функций совпадают с private-методами из `feature_processor.py`.

In [ ]:
def _add_width_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Ширина" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Ширина"], errors="coerce")
    result["lead_Ширина"] = series.fillna(series.median())


def _add_linear_height_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Линейная высота (см)" not in block_df.columns:
        return
    series = pd.to_numeric(block_df["lead_Линейная высота (см)"], errors="coerce")
    result["lead_Линейная высота (см)"] = series.fillna(series.median())
    result["lead_Линейная высота (см)__was_missing"] = series.isna().astype(int)


def _add_payment_type_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Вид оплаты" not in block_df.columns:
        return
    series = (
        block_df["lead_Вид оплаты"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
        .str.lower()
    )
    result["lead_Вид оплаты"] = series.replace({"": "unknown"}).astype(str)


def _add_returned_ts_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "returned_ts" not in block_df.columns:
        return
    ts = pd.to_datetime(block_df["returned_ts"], errors="coerce")
    result["returned_ts"] = ts.astype("string")
    result["returned_ts__is_present"] = ts.notna().astype(int)


def _add_delivery_service_feature(block_df: pd.DataFrame, result: pd.DataFrame) -> None:
    if "lead_Служба доставки" not in block_df.columns:
        return
    series = (
        block_df["lead_Служба доставки"]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )
    value_counts = series.value_counts(dropna=False)
    rare_values = value_counts[value_counts < 10].index
    result["lead_Служба доставки"] = series.replace(list(rare_values), "OTHER").astype(str)


def _add_default_feature(block_df: pd.DataFrame, result: pd.DataFrame, column: str) -> None:
    series = block_df[column]
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        result[column] = series.fillna("").astype(str)
    else:
        result[column] = series

In [ ]:
X_block = pd.DataFrame(index=block_df.index)

for column in block_df.columns:
    if column == "lead_Ширина":
        _add_width_feature(block_df, X_block)
    elif column == "lead_Линейная высота (см)":
        _add_linear_height_feature(block_df, X_block)
    elif column == "lead_Вид оплаты":
        _add_payment_type_feature(block_df, X_block)
    elif column == "returned_ts":
        _add_returned_ts_feature(block_df, X_block)
    elif column == "lead_Служба доставки":
        _add_delivery_service_feature(block_df, X_block)
    else:
        _add_default_feature(block_df, X_block, column)

print("Processed shape:", X_block.shape)
display(X_block.head())

In [ ]:
X_block.to_csv(OUTPUT_DIR / "X_block.csv", index=False)
feature_spec = create_feature_spec_template(X_block)
feature_spec.to_csv(OUTPUT_DIR / "feature_spec.csv", index=False)

print("Saved:")
print(OUTPUT_DIR / "X_block.csv")
print(OUTPUT_DIR / "feature_spec.csv")